# Tema 5, Exercițiul 2
## Putem deduce dificultatea testelor din notele obținute? (Viterbi)

Dificultatea testului este starea ascunsă $X_t \in \{D, M, U\}$ (dificil, mediu, ușor), iar nota $O_t \in \{FB, B, S, NS\}$ este observația. Implementăm direct algoritmul Viterbi pentru a găsi cea mai probabilă secvență de dificultăți conform notelor observate.

## a) Definirea modelului

- distribuția inițială $\pi_j = P(X_1 = j)$ (uniformă: dificultatea primului test e la fel de probabilă);
- matricea de tranziție $a_{ij} = P(X_{t+1} = j \mid X_t = i)$, rânduri și coloane în ordinea D, M, U;
- matricea de emisie $b_j(o) = P(O_t = o \mid X_t = j)$, coloane în ordinea FB, B, S, NS.

Din dificil următorul test e doar M sau U (câte $0.5$); din M sau U e dificil cu $0.5$, altfel M sau U cu $0.25$.

$$\pi = \begin{pmatrix} \tfrac13 & \tfrac13 & \tfrac13 \end{pmatrix}, \qquad A = \begin{pmatrix} 0 & 0.5 & 0.5 \\ 0.5 & 0.25 & 0.25 \\ 0.5 & 0.25 & 0.25 \end{pmatrix}, \qquad B = \begin{pmatrix} 0.1 & 0.2 & 0.4 & 0.3 \\ 0.15 & 0.25 & 0.5 & 0.1 \\ 0.2 & 0.3 & 0.4 & 0.1 \end{pmatrix}.$$

Fiecare rând din $A$ și din $B$ este o distribuție de probabilitate, deci trebuie să aibă suma $1$.

In [1]:
import numpy as np

stari = ["D", "M", "U"]            # dificil, mediu, usor
note = ["FB", "B", "S", "NS"]

pi = np.array([1/3, 1/3, 1/3])     # pi_j = P(X_1 = j)

# a_ij = P(X_{t+1}=j | X_t=i), ordine D, M, U
A = np.array([
    [0.0,  0.5,  0.5 ],   # din D -> M / U
    [0.5,  0.25, 0.25],   # din M -> D / M / U
    [0.5,  0.25, 0.25],   # din U -> D / M / U
])

# b_j(o) = P(O_t=o | X_t=j), coloane FB, B, S, NS
B = np.array([
    [0.1,  0.2,  0.4, 0.3],   # D
    [0.15, 0.25, 0.5, 0.1],   # M
    [0.2,  0.3,  0.4, 0.1],   # U
])

# verificam ca fiecare rand e o distributie (suma 1)
print("sume randuri A:", A.sum(axis=1))
print("sume randuri B:", B.sum(axis=1))

sume randuri A: [1. 1. 1.]
sume randuri B: [1. 1. 1.]


Codăm secvența de note observate ca indici de coloană în $B$ (FB=0, B=1, S=2, NS=3).

In [2]:
secventa = ["FB", "FB", "S", "B", "B", "S", "B", "B", "NS", "B", "B", "S"]
o = [note.index(x) for x in secventa]   # indici observatii
T, n = len(o), len(stari)
print("T =", T, " observatii codate:", o)

T = 12  observatii codate: [0, 0, 2, 1, 1, 2, 1, 1, 3, 1, 1, 2]


## b) Inițializarea Viterbi

$\delta_t(j)$ = probabilitatea comună **maximă** a unei secvențe de stări care se termină în $j$ la momentul $t$, împreună cu observațiile până la $t$:

$$\delta_t(j) = \max_{x_1, \dots, x_{t-1}} P(x_1, \dots, x_{t-1}, X_t = j,\ o_1, \dots, o_t).$$

Pentru $t = 1$: $\delta_1(j) = \pi_j\, b_j(o_1)$.

Aceasta e produsul element cu element dintre $\pi$ și coloana lui $o_1$ din $B$.

In [3]:
# delta[t, j] = scorul celui mai bun traseu care se termina in starea j la momentul t
# psi[t, j]   = starea de la momentul t-1 de pe acel traseu (pentru backtracking)
delta = np.zeros((T, n))
psi = np.zeros((T, n), dtype=int)

# initializare: delta_1(j) = pi_j * b_j(o_1)
delta[0] = pi * B[:, o[0]]

## c) Implementarea recurenței

Pentru $t = 2, \dots, T$:

$$\delta_t(j) = b_j(o_t)\,\max_i\big\{\delta_{t-1}(i)\,a_{ij}\big\}, \qquad \psi_t(j) = \arg\max_i\big\{\delta_{t-1}(i)\,a_{ij}\big\}.
$$

Cel mai bun traseu care ajunge în $j$ la momentul $t$ trebuie să treacă printr-o stare $i$ la $t-1$.

La fiecare pas $t$, pentru o stare țintă $j$, codul evaluează vectorul $\delta_{t-1}(i)\,a_{ij}$ peste toate stările precedente $i$, apoi ia maximul (pentru $\delta$) și poziția lui (pentru $\psi$):

$$\underbrace{\big[\delta_{t-1}(i)\,a_{ij}\big]_i}_{\texttt{scoruri}} \;\longrightarrow\; \delta_t(j) = b_j(o_t)\cdot\max(\texttt{scoruri}), \quad \psi_t(j) = \arg\max(\texttt{scoruri}).$$

$\delta$ ne dă doar scorul maxim, nu și drumul. $\psi_t(j)$ reține din ce stare $i$ s-a atins acel maxim.

In [4]:
for t in range(1, T):
    for j in range(n):
        # scorul fiecarei stari precedente i, daca am trece i -> j
        scoruri = delta[t-1] * A[:, j]      # delta_{t-1}(i) * a_ij
        psi[t, j] = np.argmax(scoruri)      # cea mai buna stare precedenta
        delta[t, j] = B[j, o[t]] * scoruri.max()   # inmultim cu emisia curenta

## d) Verificare intermediară

Afișăm $\delta_1$ și $\delta_2$ (după primele două note FB, FB) și verificăm manual un calcul din $\delta_2$.

Refacem de mână exact formula recurenței pentru $j = D$ (coloana D din $A$ este $[a_{DD}, a_{MD}, a_{UD}]$):

$$\delta_2(D) = b_D(o_2)\,\max_i\big\{\delta_1(i)\,a_{iD}\big\}.$$

In [5]:
print("delta_1:", dict(zip(stari, delta[0])))
print("delta_2:", dict(zip(stari, delta[1])))

# verificare manuala delta_2(D) = b_D(FB) * max_i [ delta_1(i) * a_iD ]
candidati = [delta[0, i] * A[i, 0] for i in range(n)]
manual_D = B[0, o[1]] * max(candidati)
print("\ncandidati delta_1(i)*a_iD:", dict(zip(stari, candidati)))
print("delta_2(D) manual:", manual_D, " din tabel:", delta[1, 0])

delta_1: {'D': np.float64(0.03333333333333333), 'M': np.float64(0.049999999999999996), 'U': np.float64(0.06666666666666667)}
delta_2: {'D': np.float64(0.0033333333333333335), 'M': np.float64(0.0025), 'U': np.float64(0.0033333333333333335)}

candidati delta_1(i)*a_iD: {'D': np.float64(0.0), 'M': np.float64(0.024999999999999998), 'U': np.float64(0.03333333333333333)}
delta_2(D) manual: 0.0033333333333333335  din tabel: 0.0033333333333333335


Verificare: cu $o_2 = FB$, $b_D(FB) = 0.1$ și coloana D din $A$ este $[0,\ 0.5,\ 0.5]$, deci $\delta_2(D) = 0.1 \cdot \max\{0,\ 0.5 \cdot \tfrac{0.15}{3},\ 0.5 \cdot \tfrac{0.2}{3}\} = 0.1 \cdot \tfrac{0.1}{3} \approx 0.00333$, egal cu valoarea din tabel.

## e) Reconstrucția traseului

Citim din $\delta$ și $\psi$ secvența optimă. Ultima coloană $\delta_T$ conține scorul celui mai bun traseu care se termină în fiecare stare, deci starea finală optimă este cea cu scor maxim:

$$x_T^* = \arg\max_j \delta_T(j), \qquad \max_{x_1, \dots, x_T} P(x_1, \dots, x_T,\ o_1, \dots, o_T) = \max_j \delta_T(j).$$

De la ea mergem înapoi urmând pointerii $\psi$: fiecare $\psi_t(j)$ ne spune din ce stare am ajuns optim în $j$, deci

$$x_{t-1}^* = \psi_t(x_t^*), \qquad t = T, T-1, \dots, 2.$$

Parcurgând această recurență de la $T$ spre $1$ reconstituim întreaga secvență de 12 dificultăți, iar scorul $\max_j \delta_T(j)$ este chiar probabilitatea comună a traseului optim și a observațiilor.

In [6]:
# starea finala cu scor maxim
traseu = np.zeros(T, dtype=int)
traseu[T-1] = np.argmax(delta[T-1])

# backtracking: reconstruim traseul in ordine inversa folosind psi
for t in range(T-2, -1, -1):
    traseu[t] = psi[t+1, traseu[t+1]]

dificultati = [stari[i] for i in traseu]
prob_comuna = delta[T-1].max()

for nota, dif in zip(secventa, dificultati):
    print(f"{nota:>3}  ->  {dif}")
print("\nsecventa de dificultati:", dificultati)
print("max P(x_1..x_12, o_1..o_12) =", prob_comuna)

 FB  ->  D
 FB  ->  U
  S  ->  D
  B  ->  U
  B  ->  D
  S  ->  M
  B  ->  D
  B  ->  U
 NS  ->  D
  B  ->  U
  B  ->  D
  S  ->  M

secventa de dificultati: ['D', 'U', 'D', 'U', 'D', 'M', 'D', 'U', 'D', 'U', 'D', 'M']
max P(x_1..x_12, o_1..o_12) = 2.1093750000000005e-11


## f) Interpretare

Viterbi caută cea mai probabilă secvență completă de dificultăți, nu cea mai probabilă dificultate separat la fiecare test.

Alegerea stării optime individual la fiecare moment (MAP) poate produce o secvență cu tranziții imposibile.

De exemplu, fiindcă $a_{DD} = 0$, niciun traseu ales de Viterbi nu poate avea două teste dificile consecutive.

Scorul de la punctul e) este o probabilitate comună $P(x^*, o)$, nu probabilitatea condiționată a traseului dat fiind șirul notelor.